In [6]:
#import cell
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from prettytable import PrettyTable
from utils import stats, exploration_plots, preprocess_basilar

**importing the data**

In [7]:
df = pd.read_csv("development.csv", header = "infer")
evaluate_df = pd.read_csv("evaluation.csv", header = "infer")

**Starting the data cleaning**

In [8]:
# missing_per_feature = df.isnull().sum()
# print(missing_per_feature)

missing_per_feature_eval = evaluate_df.isnull().sum()
print(missing_per_feature_eval)

category            0
title               0
body                0
amenities        3240
bathrooms          11
bedrooms           34
currency            0
fee                 0
has_photo           0
pets_allowed    12102
price_type          0
square_feet         0
address         18320
cityname           68
state              68
latitude            2
longitude           2
source              0
time                0
id                  0
dtype: int64


In [9]:
df = preprocess_basilar(df)
evaluate_df = preprocess_basilar(evaluate_df,is_eval=True)

In [ ]:
stats(df)
stats(evaluate_df)
#now i want to explore the relationships between the features and the price:

In [ ]:
print(f'max square feet : {df["square_feet"].max()}, min square feet : {df["square_feet"].min()}' )

In [ ]:
features = ["bathrooms", "bedrooms", "has_photo", "pets_allowed", "square_feet", "time", "state"]
exploration_plots(df, features)
exploration_plots(evaluate_df,features)

In [ ]:
plt.figure(figsize=(24, 5))
plt.scatter(df["square_feet"], df["price"], alpha=0.5)
plt.xlabel("square_feet")
plt.ylabel("price")
plt.title(f"square_feet vs Price (Scatter)")

In [10]:
for col in ["title", "body", "amenities"]:
    if col in df.columns:
        df = df.drop(columns=col)
        evaluate_df = evaluate_df.drop(columns=col)



In [13]:
y = df["price"]
df = df.drop(columns="price")
# Encode all categorical columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
X_encoded = pd.get_dummies(df, columns=categorical_cols)
X_test_encoded = pd.get_dummies(evaluate_df,columns=categorical_cols)

In [ ]:
# Separate features and target
table = PrettyTable()
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

rfr = RandomForestRegressor()
rfr.fit(X_train, y_train)
y_pred = rfr.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
table.field_names = ["Model", "value"]
table.add_row(["Random Forest", mae])
print(table)


In [25]:
importances = rfr.feature_importances_
feature_names = X_train.columns
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)
print(feat_imp.head(20))  # Show top 20 most important features

square_feet                   0.243158
longitude                     0.174800
state_CA                      0.125757
latitude                      0.097181
time                          0.064256
bathrooms                     0.055662
bedrooms                      0.040578
id                            0.033690
cityname_Barstow              0.023723
cityname_New York             0.012122
cityname_West Hollywood       0.010008
cityname_Pacific Palisades    0.006985
state_CO                      0.006978
state_WA                      0.006723
cityname_Jersey City          0.005157
cityname_Glasco               0.003884
cityname_Chicago              0.003189
cityname_Atlanta              0.003064
has_photo_Thumbnail           0.002809
has_photo_Yes                 0.002781
dtype: float64


In [ ]:
rfr_online = RandomForestRegressor()
rfr_online.fit(X_encoded,y)
# Align columns of X_test_encoded to X_encoded
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)
# Now you can safely predict
y_pred = rfr_online.predict(X_test_encoded)
results_df = pd.DataFrame({
    "Id": range(len(y_pred)),
    "Predicted": y_pred.astype(float)  # or round if you want integers
})

# Save to CSV
results_df.to_csv("predictions.csv", index=False)

da fare
- assegnazioni valori a bathrooms and bedrooms
- trovare valori states tramite longitute e latitude più vicine
- distribuzione stati nel dataset (evaluation e train)
- gestire i campi title e body con tf-Idf
